# 🧪 EXERCISE: Longitudinal Gaze Metrics

> **Instructions:** This notebook has intentional gaps — paths, loops, calculations and plot labels that you must complete. Look for `# TODO` and `???` markers. Run each cell in order once you have filled in the blanks.

## Learning objectives

- Configure session paths for multi-session analysis
- Compute accuracy and descriptive statistics from trial-level data
- Build a bar chart comparing saccade metrics across sessions using a loop
- Export the combined trial metrics DataFrame to CSV

---

# VISION Task — Longitudinal Gaze Metrics (One Patient, All Sessions)

Tracks eye-movement metrics across multiple recording sessions for a single patient.  
Loads all session folders automatically from a base directory.

| Metric class | Measures |
|---|---|
| **Fixations** | Count, mean duration, total fixation time, dispersion |
| **Saccades** | Count, amplitude, peak velocity, latency to first saccade |
| **Blinks** | Count, mean duration, rate |
| **Gaze at response** | Where gaze was at key press vs target |
| **Longitudinal** | Session-to-session changes, trend tests |

**SCOPE: one participant × all sessions**

---
## 0 Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from pathlib import Path
import msgpack, struct

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.15)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})

# ── TODO 1: Set participant and session paths ─────────────────────────────────
PARTICIPANT_ID = '???'   # e.g. 'JF'

# Each tuple: (session_label, path_to_recording_folder)
# Add one entry per session you want to analyse.
SESSIONS = [
    # ('S001', Path('???')),   # ← Fill in the real path
    # ('S002', Path('???')),
]

GAZE_TOPIC               = 'gaze.3d.01.'
VELOCITY_THRESHOLD_DEG_S = 30
MIN_SACCADE_DUR_S        = 0.010
MAX_SACCADE_DUR_S        = 0.100
BLINK_GROUP_GAP_S        = 0.075
POSITION_STEP_DEG        = 4.5


---
## 1 Shared Helper Functions

In [ ]:
def load_pldata(path):
    """Load a Pupil Labs .pldata file → list of (topic, dict) tuples."""
    records = []
    with open(path, 'rb') as f:
        unpacker = msgpack.Unpacker(f, raw=False, strict_map_key=False)
        for item in unpacker:
            if isinstance(item, (list, tuple)) and len(item) == 2:
                topic, payload = item
                rec = payload if isinstance(payload, dict) else (
                    msgpack.unpackb(payload, raw=False, strict_map_key=False)
                    if isinstance(payload, bytes) else None
                )
                if rec is not None:
                    records.append((str(topic), rec))
    return records


def parse_onset_label(label):
    """Parse VISION_task onset annotation."""
    m = re.match(r'trial_(\d+)_onset_ax(\d+)_pos(\d+)([LR])_(\w+)', label)
    if m:
        return {
            'trial_num'   : int(m.group(1)),
            'axis_deg'    : int(m.group(2)),
            'position_num': int(m.group(3)),
            'side'        : m.group(4),
            'shape'       : m.group(5),
        }
    return None


def parse_offset_label(label):
    """Parse offset annotation → dict with trial_num and outcome."""
    m = re.match(r'trial_(\d+)_offset_(\w+)', label)
    if m:
        return {'trial_num': int(m.group(1)), 'outcome': m.group(2)}
    return None


def detect_saccades(gaze_df, vel_thresh=VELOCITY_THRESHOLD_DEG_S, 
                    min_dur_ms=SACCADE_MIN_DURATION_MS,
                    max_dur_ms=SACCADE_MAX_DURATION_MS):
    """Detect saccades from gaze velocity."""
    if len(gaze_df) < 3:
        return pd.DataFrame()
    
    # Smooth gaze
    sr = 1.0 / np.median(np.diff(gaze_df['timestamp'].values))
    sigma_samples = max(1, int(0.005 * sr))
    xs = gaussian_filter1d(gaze_df['x_deg'].fillna(0).values, sigma_samples)
    ys = gaussian_filter1d(gaze_df['y_deg'].fillna(0).values, sigma_samples)
    
    # Velocity
    dt = np.diff(gaze_df['timestamp'].values)
    vel = np.hypot(np.diff(xs), np.diff(ys)) / np.where(dt > 0, dt, 1e-6)
    vel = np.append(vel, vel[-1])
    
    # Binary: above threshold
    above = vel >= vel_thresh
    
    # Find saccade intervals
    edges = np.diff(above.astype(int))
    starts = np.where(edges == 1)[0]
    ends = np.where(edges == -1)[0]
    
    if len(starts) == 0:
        return pd.DataFrame()
    # If signal starts above threshold, the first end precedes the first start — drop it
    if len(ends) > 0 and ends[0] < starts[0]:
        ends = ends[1:]
    if len(ends) < len(starts):
        ends = np.append(ends, len(gaze_df) - 1)
    
    rows = []
    for i_start, i_end in zip(starts, ends[:len(starts)]):
        t_start = gaze_df.iloc[i_start]['timestamp']
        t_end = gaze_df.iloc[i_end]['timestamp']
        dur_ms = (t_end - t_start) * 1000
        
        if not (min_dur_ms <= dur_ms <= max_dur_ms):
            continue
        
        x_start, y_start = xs[i_start], ys[i_start]
        x_end, y_end = xs[i_end], ys[i_end]
        amp_deg = np.hypot(x_end - x_start, y_end - y_start)
        peak_vel = vel[i_start:i_end+1].max()
        
        rows.append({
            'timestamp'     : t_start,
            'start_x'       : x_start,
            'start_y'       : y_start,
            'end_x'         : x_end,
            'end_y'         : y_end,
            'amplitude_deg' : amp_deg,
            'peak_vel_deg_s': peak_vel,
            'duration_ms'   : dur_ms,
        })
    
    return pd.DataFrame(rows)


def build_blink_table(blink_df):
    """Group blinks with small gaps into single events."""
    if len(blink_df) == 0:
        return pd.DataFrame(columns=['timestamp', 'duration_ms'])
    
    blink_df = blink_df.sort_values('timestamp').reset_index(drop=True)
    grouped_onsets = []
    current_group = [blink_df['timestamp'].iloc[0]]
    
    for i in range(1, len(blink_df)):
        gap = blink_df['timestamp'].iloc[i] - current_group[-1]
        if gap <= BLINK_GROUP_GAP_S:
            current_group.append(blink_df['timestamp'].iloc[i])
        else:
            grouped_onsets.append(current_group[0])
            current_group = [blink_df['timestamp'].iloc[i]]
    grouped_onsets.append(current_group[0])
    
    off_ts = blink_df['timestamp'].values
    rows = []
    for go in grouped_onsets:
        candidates = off_ts[off_ts > go]
        if len(candidates):
            t_off = candidates[0]
            rows.append({
                'timestamp'     : go,
                'end_timestamp' : t_off,
                'duration_ms'   : (t_off - go) * 1000,
            })
    return pd.DataFrame(rows)


def events_in_window(df, t_start, t_end, ts_col='timestamp'):
    return df[(df[ts_col] >= t_start) & (df[ts_col] <= t_end)]


def trial_fixation_metrics(fix_in):
    if len(fix_in) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }
    fix_valid = fix_in[
        (fix_in['duration_ms'] >= MIN_FIX_DURATION_MS) &
        (fix_in['confidence']  >= MIN_GAZE_CONFIDENCE)
    ]
    if len(fix_valid) == 0:
        return {
            'fix_count': 0,
            'fix_dur_mean_ms': np.nan,
            'fix_dur_total_ms': 0,
            'fix_dispersion_mean': np.nan
        }
    return {
        'fix_count': len(fix_valid),
        'fix_dur_mean_ms': fix_valid['duration_ms'].mean(),
        'fix_dur_total_ms': fix_valid['duration_ms'].sum(),
        'fix_dispersion_mean': (
            fix_valid['dispersion'].mean()
            if 'dispersion' in fix_valid else np.nan
        ),
    }


def trial_saccade_metrics(sacc_in, t_start, target_x=None, target_y=None):
    if len(sacc_in) == 0:
        return {'sacc_count': 0, 'sacc_amp_mean_deg': np.nan,
                'sacc_peak_vel_mean': np.nan, 'sacc_dur_mean_ms': np.nan,
                'sacc_latency_ms': np.nan}
    metrics = {
        'sacc_count'       : len(sacc_in),
        'sacc_amp_mean_deg': sacc_in['amplitude_deg'].mean(),
        'sacc_peak_vel_mean': sacc_in['peak_vel_deg_s'].mean(),
        'sacc_dur_mean_ms' : sacc_in['duration_ms'].mean(),
        'sacc_latency_ms'  : (sacc_in['timestamp'].min() - t_start) * 1000,
    }
    if target_x is not None and len(sacc_in):
        first = sacc_in.sort_values('timestamp').iloc[0]
        dist_start = np.hypot(first['start_x'] - target_x, first['start_y'] - target_y)
        dist_end   = np.hypot(first['end_x']   - target_x, first['end_y']   - target_y)
        metrics['first_sacc_toward_target'] = int(dist_end < dist_start)
    return metrics


def trial_blink_metrics(blink_in):
    if len(blink_in) == 0:
        return {'blink_count': 0, 'blink_dur_mean_ms': np.nan}
    return {
        'blink_count': len(blink_in),
        'blink_dur_mean_ms': blink_in['duration_ms'].mean() if 'duration_ms' in blink_in else np.nan,
    }


def gaze_at_response(gaze_in, t_end, window_s=0.15):
    """Return median gaze position in the 150 ms window just before key press."""
    if t_end is None or np.isnan(t_end):
        return None
    window = gaze_in[
        (gaze_in['timestamp'] >= t_end - window_s) &
        (gaze_in['timestamp'] <= t_end) &
        (gaze_in['confidence'] >= MIN_GAZE_CONFIDENCE) &
        (gaze_in['norm_x'].between(0.0, 1.0)) &
        (gaze_in['norm_y'].between(0.0, 1.0))
    ]
    if len(window) == 0:
        return None
    return {
        'x_deg'     : window['x_deg'].median(),
        'y_deg'     : window['y_deg'].median(),
        'confidence': window['confidence'].mean(),
    }


def get_target_xy(axis_deg, position_num, side_char):
    """Compute target position from task parameters."""
    side  = 1 if side_char == 'R' else -1
    ecc   = POSITIONS_DEG[position_num - 1]
    a_rad = np.radians(axis_deg)
    return side * ecc * np.cos(a_rad), side * ecc * np.sin(a_rad)


---
## 2 Load All Sessions

In [ ]:
def load_session(recording_dir, session_label):
    """Load one session folder → (gaze_df, fix_df, blink_df, ann_df, trials, trial_metrics)"""
    recording_dir = Path(recording_dir)
    result = {'label': session_label, 'dir': recording_dir}
    
    # Load .pldata files
    try:
        gaze_raw   = load_pldata(recording_dir / 'gaze.pldata')
        fix_raw    = load_pldata(recording_dir / 'fixations.pldata')
        blink_raw  = load_pldata(recording_dir / 'blinks.pldata')
        ann_raw    = load_pldata(recording_dir / 'annotation.pldata')
    except FileNotFoundError as e:
        print(f'  [{session_label}] Missing file: {e}')
        return None

    def is_valid_norm(rec):
        norm = rec.get('norm_pos', [0, 0])
        return -0.05 <= norm[0] <= 1.05 and -0.05 <= norm[1] <= 1.05

    # Gaze
    gaze_rows = []
    for topic, rec in gaze_raw:
        if not topic.startswith('gaze.'): continue
        if not is_valid_norm(rec): continue
        norm = rec.get('norm_pos', [0.5, 0.5])
        gaze_rows.append({
            'timestamp' : rec.get('timestamp', np.nan),
            'confidence': rec.get('confidence', 0.0),
            'norm_x'    : norm[0],
            'norm_y'    : norm[1],
            'x_deg'     : (norm[0] - 0.5) * SCREEN_W_DEG,
            'y_deg'     : (norm[1] - 0.5) * SCREEN_H_DEG,
        })
    gaze_df = pd.DataFrame(gaze_rows).sort_values('timestamp').reset_index(drop=True)

    # Fixations
    fix_rows = []
    for topic, rec in fix_raw:
        if not topic.startswith('fixations'): continue
        norm = rec.get('norm_pos', [0.5, 0.5])
        raw_dur = rec.get('duration', 0.0)
        dur_ms =  raw_dur * 1000 if 10 > raw_dur > 0.1 else raw_dur
        fix_rows.append({
            'timestamp'   : rec.get('timestamp', np.nan),
            'confidence'  : rec.get('confidence', 0.0),
            'norm_x'      : norm[0],
            'norm_y'      : norm[1],
            'x_deg'       : (norm[0] - 0.5) * SCREEN_W_DEG,
            'y_deg'       : (norm[1] - 0.5) * SCREEN_H_DEG,
            'duration_ms' : dur_ms,
            'dispersion'  : rec.get('dispersion', np.nan),
        })
    fix_df = pd.DataFrame(fix_rows)
    if not fix_df.empty:
        fix_df = fix_df.sort_values('timestamp').reset_index(drop=True)

    # Blinks
    blink_rows = []
    for topic, rec in blink_raw:
        blink_rows.append({
            'timestamp': rec.get('timestamp', np.nan),
            'type'     : rec.get('type', ''),
            'duration' : rec.get('duration', np.nan),
        })
    blink_df = pd.DataFrame(blink_rows).sort_values('timestamp').reset_index(drop=True)
    blinks = build_blink_table(blink_df)

    # Annotations
    ann_rows = []
    for topic, rec in ann_raw:
        ann_rows.append({
            'timestamp': rec.get('timestamp', np.nan),
            'label'    : rec.get('label', ''),
        })
    ann_df = pd.DataFrame(ann_rows).sort_values('timestamp').reset_index(drop=True)

    # Parse trials
    onsets  = ann_df[ann_df['label'].str.contains('_onset_', na=False)].copy()
    offsets = ann_df[ann_df['label'].str.contains('_offset_', na=False)].copy()
    trial_list = []
    for _, row in onsets.iterrows():
        parsed = parse_onset_label(row['label'])
        if parsed:
            parsed['t_start'] = row['timestamp']
            trial_list.append(parsed)
    trials = pd.DataFrame(trial_list)
    if len(trials) == 0:
        print(f'  [{session_label}] No trials found')
        return None
    
    # Add end times
    offset_lookup = {}
    for _, row in offsets.iterrows():
        parsed = parse_offset_label(row['label'])
        if parsed:
            tn = parsed['trial_num']
            if tn not in offset_lookup:
                offset_lookup[tn] = {'t_end': row['timestamp'], 'outcome': parsed['outcome']}
    trials['t_end']   = trials['trial_num'].map(lambda x: offset_lookup.get(x, {}).get('t_end', np.nan))
    trials['outcome'] = trials['trial_num'].map(lambda x: offset_lookup.get(x, {}).get('outcome', 'unknown'))

    # Per-trial metrics
    sacc_df = detect_saccades(gaze_df)
    metrics_list = []
    for _, trial in trials.iterrows():
        t0, t1 = trial['t_start'], trial.get('t_end', np.nan)
        if np.isnan(t1): continue
        fix_in   = events_in_window(fix_df, t0, t1)
        sacc_in  = events_in_window(sacc_df, t0, t1) if len(sacc_df) else pd.DataFrame()
        blink_in = events_in_window(blinks, t0, t1) if len(blinks) else pd.DataFrame()
        gaze_resp= gaze_at_response(gaze_df, t1)
        tx, ty   = get_target_xy(trial['axis_deg'], trial['position_num'], trial['side'])
        
        row_m = {
            'session'       : session_label,
            'trial_num'     : trial['trial_num'],
            'axis_deg'      : trial['axis_deg'],
            'position_num'  : trial['position_num'],
            'side'          : trial['side'],
            'shape'         : trial.get('shape', ''),
            'outcome'       : trial['outcome'],
            't_start'       : t0,
            't_end'         : t1,
            'duration_s'    : t1 - t0,
            'target_x_deg'  : tx,
            'target_y_deg'  : ty,
            'eccentricity_deg': POSITION_STEP_DEG * trial['position_num'],
            'axis_label'    : {0: 'Horizontal', 45: 'Diagonal BL→TR', 135: 'Diagonal TL→BR'}.get(trial['axis_deg'], str(trial['axis_deg'])),
            'hemifield'     : 'Right' if trial['side'] == 'R' else 'Left',
        }
        row_m.update(trial_fixation_metrics(fix_in))
        row_m.update(trial_saccade_metrics(sacc_in, t0, tx, ty))
        row_m.update(trial_blink_metrics(blink_in))
        if gaze_resp is not None:
            row_m['resp_gaze_x'] = gaze_resp['x_deg']
            row_m['resp_gaze_y'] = gaze_resp['y_deg']
            row_m['resp_gaze_conf'] = gaze_resp['confidence']
            row_m['gaze_error_deg'] = np.hypot(gaze_resp['x_deg'] - tx, gaze_resp['y_deg'] - ty)
        metrics_list.append(row_m)

    trial_metrics = pd.DataFrame(metrics_list)
    print(f'  [{session_label}] {len(trials)} trials, {len(trial_metrics)} with metrics')
    return {
        'label': session_label,
        'gaze_df': gaze_df, 'fix_df': fix_df, 'blinks': blinks,
        'ann_df': ann_df, 'trials': trials, 'trial_metrics': trial_metrics,
        'sacc_df': sacc_df,
    }

# Load all sessions
print(f'\nLoading {len(SESSION_DIRS)} sessions for {PARTICIPANT_ID}...')
sessions = []
for lbl, d in zip(SESSION_LABELS, SESSION_DIRS):
    s = load_session(d, lbl)
    if s: sessions.append(s)

print(f'\nLoaded {len(sessions)} sessions successfully')
all_tm = pd.concat([s['trial_metrics'] for s in sessions], ignore_index=True)
print(f'Total trials across all sessions: {len(all_tm)}')


---
## 3 Session-Level Summary Table

In [ ]:
# ── TODO 2: Build the session summary table ───────────────────────────────────
# Loop over `sessions` (list of dicts returned by load_session).
# For each session, compute:
#   - n           : total number of trials
#   - n_corr      : number of correct trials (outcome == 'correct')
#   - accuracy    : n_corr / n  (as a percentage)
#   - lat_mean    : mean of sacc_latency_ms
#   - lat_sd      : standard deviation of sacc_latency_ms
#   - amp_mean    : mean of sacc_amp_mean_deg
# Append a dict to `rows` and print the resulting DataFrame.

rows = []
for s in sessions:
    tm  = s['trial_metrics']
    lbl = s['label']

    # Your code here — replace the ??? placeholders:
    n      = ???
    n_corr = ???
    acc    = ???

    rows.append({
        'Session'       : lbl,
        'N trials'      : n,
        'Accuracy (%)'  : round(acc * 100, 1),
        'Sacc latency (ms)': f"{tm['sacc_latency_ms'].mean():.1f} ± {tm['sacc_latency_ms'].std():.1f}",
        'Sacc amp (°)'  : f"{tm['sacc_amp_mean_deg'].mean():.2f} ± {tm['sacc_amp_mean_deg'].std():.2f}",
    })

summary_df = pd.DataFrame(rows).set_index('Session')
print(f'Session Summary — {PARTICIPANT_ID}')
print(summary_df.to_string())


---
## 4 Saccade Metrics Over Sessions

In [ ]:
sacc_metrics = ['sacc_count', 'sacc_amp_mean_deg', 'sacc_peak_vel_mean', 'sacc_latency_ms']
sacc_labels  = ['Saccade count / trial', 'Mean amplitude (°)',
                'Mean peak velocity (°/s)', 'First saccade latency (ms)']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for ax, metric, label in zip(axes, sacc_metrics, sacc_labels):
    # ── TODO 3: Plot one bar per session ─────────────────────────────────────
    # For each session in `sessions`:
    #   - extract the metric column from s['trial_metrics']
    #   - compute the mean and SEM  (SEM = std / sqrt(n))
    #   - plot a bar at position i (use ax.bar) with an error bar (yerr=sem)
    #   - label the bar with the session label
    #
    # Hint: loop with enumerate(sessions), use ax.set_xticks and ax.set_xticklabels

    # Your code here:
    pass   # ← replace this

    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle(f'{PARTICIPANT_ID} — Saccade Metrics Across Sessions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 5 Blink Metrics Over Sessions

In [ ]:
blink_metrics = ['blink_count', 'blink_dur_mean_ms']
blink_labels  = ['Blinks per trial', 'Mean blink duration (ms)']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, label in zip(axes, blink_metrics, blink_labels):
    means, sds, lbls = [], [], []
    for s in sessions:
        vals = s['trial_metrics'][col].dropna() if col in s['trial_metrics'] else pd.Series([])
        means.append(vals.mean()); sds.append(vals.std()); lbls.append(s['label'])
    ax.errorbar(range(len(lbls)), means, yerr=sds, fmt='^-', capsize=5,
                color='purple', lw=2, ms=7)
    x = np.array(range(len(means)))
    valid = [(xi, m) for xi, m in zip(x, means) if not np.isnan(m)]
    if len(valid) >= 2:
        xv = np.array([v[0] for v in valid]); yv = np.array([v[1] for v in valid])
        sl, ic, r, p, _ = stats.linregress(xv, yv)
        ax.plot(x, sl*x+ic, 'r--', lw=1.2, label=f'Trend r={r:.2f}, p={p:.3f}')
        ax.legend(fontsize=8)
    ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=45)
    ax.set_ylabel(label); ax.set_title(label)

fig.suptitle(f'{PARTICIPANT_ID} — Blink Metrics Across Sessions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 6 Gaze Error at Key Press Over Sessions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: mean gaze error per session
ax = axes[0]
means, sds, lbls = [], [], []
for s in sessions:
    vals = s['trial_metrics']['gaze_error_deg'].dropna() if 'gaze_error_deg' in s['trial_metrics'] else pd.Series([])
    means.append(vals.mean()); sds.append(vals.std()); lbls.append(s['label'])
ax.errorbar(range(len(lbls)), means, yerr=sds, fmt='D-', capsize=5,
            color='crimson', lw=2, ms=7)
x = np.array(range(len(means)))
valid = [(xi, m) for xi, m in zip(x, means) if not np.isnan(m)]
if len(valid) >= 2:
    xv = np.array([v[0] for v in valid]); yv = np.array([v[1] for v in valid])
    sl, ic, r, p, _ = stats.linregress(xv, yv)
    ax.plot(x, sl*x+ic, 'r--', lw=1.2, label=f'Trend r={r:.2f}, p={p:.3f}')
    ax.legend(fontsize=9)
ax.set_xticks(range(len(lbls))); ax.set_xticklabels(lbls, rotation=45)
ax.set_ylabel('Gaze error (°)'); ax.set_title('Mean gaze error at key press')

# Panel B: box plot per session
ax = axes[1]
data_boxes = [s['trial_metrics']['gaze_error_deg'].dropna().values for s in sessions
              if 'gaze_error_deg' in s['trial_metrics']]
if data_boxes:
    bp = ax.boxplot(data_boxes, labels=[s['label'] for s in sessions], patch_artist=True)
    for patch in bp['boxes']:
        patch.set_facecolor('lightsalmon')
    ax.set_ylabel('Gaze error (°)'); ax.set_title('Gaze error distribution by session')
    plt.setp(ax.get_xticklabels(), rotation=45)

fig.suptitle(f'{PARTICIPANT_ID} — Gaze Accuracy Over Sessions', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 7 Metrics by Axis — Session Comparison

In [ ]:
metrics_to_compare = [
    ('fix_count',         'Fixation count / trial'),
    ('sacc_latency_ms',   'Saccade latency (ms)'),
    ('gaze_error_deg',    'Gaze error (°)'),
    ('duration_s',        'Response time (s)'),
]
axis_order  = ['Horizontal', 'Diagonal BL→TR', 'Diagonal TL→BR']
colors = plt.cm.tab10(np.linspace(0, 0.6, len(sessions)))

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()
for ax_i, (col, label) in enumerate(metrics_to_compare):
    ax = axes[ax_i]
    x = np.arange(len(axis_order))
    w = 0.8 / max(len(sessions), 1)
    for si, s in enumerate(sessions):
        tm = s['trial_metrics']
        if col not in tm.columns: continue
        vals = tm.groupby('axis_label')[col].mean().reindex(axis_order)
        offset = (si - len(sessions)/2 + 0.5) * w
        ax.bar(x + offset, vals.values, width=w, label=s['label'],
               color=colors[si], alpha=0.8, edgecolor='k', linewidth=0.5)
    ax.set_xticks(x); ax.set_xticklabels(axis_order, rotation=20)
    ax.set_ylabel(label); ax.set_title(f'{label} by axis')
    ax.legend(fontsize=8, title='Session')

fig.suptitle(f'{PARTICIPANT_ID} — Axis-Level Metrics by Session', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


---
## 8 Statistical Tests — Longitudinal

In [ ]:
print('='*60)
print(f'LONGITUDINAL STATISTICAL TESTS — {PARTICIPANT_ID}')
print('='*60)

for col, label in [
    ('fix_count',       'Fixation count'),
    ('sacc_latency_ms', 'Saccade latency (ms)'),
    ('gaze_error_deg',  'Gaze error (°)'),
    ('duration_s',      'Response time (s)'),
]:
    print(f'\n── {label} ──────────────────────────────────────────')
    session_vals = [s['trial_metrics'][col].dropna().values
                    for s in sessions if col in s['trial_metrics'].columns]
    session_vals = [v for v in session_vals if len(v) >= 3]
    if len(session_vals) < 2:
        print('  Not enough data'); continue

    # Spearman trend (session index vs session mean)
    sess_means = np.array([v.mean() for v in session_vals])
    rho, p_rho = stats.spearmanr(range(len(sess_means)), sess_means)
    print(f'  Spearman trend: ρ={rho:.3f}, p={p_rho:.4f}')
    direction = 'improving ↓' if rho < 0 else 'worsening ↑' if rho > 0 else 'stable'
    print(f'  Direction: {direction}')

    # Friedman test (if ≥3 sessions, equal length possible)
    if len(session_vals) >= 3:
        min_len = min(len(v) for v in session_vals)
        truncated = [v[:min_len] for v in session_vals]
        try:
            stat, p_f = stats.friedmanchisquare(*truncated)
            print(f'  Friedman test: χ²={stat:.3f}, p={p_f:.4f}')
        except Exception as e:
            print(f'  Friedman: {e}')

    # First vs last session Wilcoxon (if ≥2 sessions)
    if len(session_vals) >= 2:
        v1, v2 = session_vals[0], session_vals[-1]
        min_len = min(len(v1), len(v2))
        try:
            stat_w, p_w = stats.wilcoxon(v1[:min_len], v2[:min_len])
            print(f'  Wilcoxon S1 vs S{len(session_vals)}: W={stat_w:.1f}, p={p_w:.4f}')
        except Exception as e:
            print(f'  Wilcoxon: {e}')


---
## 9 Export

In [ ]:
# ── TODO 4: Export the combined trial metrics CSV ────────────────────────────
# `all_tm` is a DataFrame containing trial metrics from all sessions combined.
# Set the output path and save it.

out_path = f'???'   # e.g. f'{PARTICIPANT_ID}_VISION_longitudinal_gaze_metrics.csv'

all_tm.to_csv(out_path, index=False)
print(f'Saved: {out_path}  ({len(all_tm)} rows, {all_tm["session"].nunique()} sessions)')
